# Task 36: k-Nearest Neighbors From Scratch

This notebook scales features, builds a balanced KD-tree, prunes distant branches, and resolves equal class votes by distance and a deterministic final rule.

In [2]:
from pathlib import Path
import heapq
import numpy as np
import pandas as pd

candidates = [Path("heart.csv"), Path("Task_36_K_Nearest_Neighbors_From_Scratch/heart.csv"), Path("Heart_Disease_Tasks/Task_36_K_Nearest_Neighbors_From_Scratch/heart.csv")]
csv_path = next(path for path in candidates if path.is_file())
data = pd.read_csv(csv_path).drop_duplicates().reset_index(drop=True)
feature_names = [name for name in ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach", "exang", "oldpeak", "slope", "ca", "thal"] if name in data.columns]
X = data[feature_names].to_numpy(dtype=float)
y = data["target"].to_numpy(dtype=int)

rng = np.random.default_rng(36)
train_parts, test_parts = [], []
for label in np.unique(y):
    indices = np.flatnonzero(y == label)
    rng.shuffle(indices)
    split_at = int(round(len(indices) * 0.8))
    train_parts.append(indices[:split_at])
    test_parts.append(indices[split_at:])
train_idx, test_idx = np.concatenate(train_parts), np.concatenate(test_parts)
rng.shuffle(train_idx)
rng.shuffle(test_idx)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
means, scales = X_train.mean(axis=0), X_train.std(axis=0)
scales[scales == 0] = 1.0
X_train, X_test = (X_train - means) / scales, (X_test - means) / scales


class KDTree:
    def __init__(self, points, labels, leaf_size=16):
        self.points = np.asarray(points, dtype=float)
        self.labels = np.asarray(labels, dtype=int)
        self.leaf_size = leaf_size
        self.root = self._build(np.arange(len(self.points)))

    def _build(self, indices):
        subset = self.points[indices]
        low, high = subset.min(axis=0), subset.max(axis=0)
        node = {"low": low, "high": high}
        if len(indices) <= self.leaf_size:
            node["indices"] = indices
            return node
        axis = int(np.argmax(high - low))
        order = indices[np.argsort(self.points[indices, axis], kind="mergesort")]
        middle = len(order) // 2
        node["left"] = self._build(order[:middle])
        node["right"] = self._build(order[middle:])
        return node

    @staticmethod
    def _box_distance_squared(point, node):
        delta = np.maximum(np.maximum(node["low"] - point, point - node["high"]), 0.0)
        return float(delta @ delta)

    def query(self, point, k):
        if not 1 <= k <= len(self.points):
            raise ValueError("k must be between 1 and the number of training rows")
        heap = []
        self.nodes_visited = 0

        def visit(node):
            self.nodes_visited += 1
            if len(heap) == k and self._box_distance_squared(point, node) > -heap[0][0]:
                return
            if "indices" in node:
                for index in node["indices"]:
                    difference = point - self.points[index]
                    distance_squared = float(difference @ difference)
                    entry = (-distance_squared, -int(index), int(index))
                    if len(heap) < k:
                        heapq.heappush(heap, entry)
                    elif entry > heap[0]:
                        heapq.heapreplace(heap, entry)
                return
            children = [node["left"], node["right"]]
            children.sort(key=lambda child: self._box_distance_squared(point, child))
            visit(children[0])
            visit(children[1])

        visit(self.root)
        neighbors = sorted(heap, key=lambda item: (-item[0], item[2]))
        indices = np.array([item[2] for item in neighbors], dtype=int)
        distances = np.sqrt(np.array([-item[0] for item in neighbors]))
        return indices, distances


def classify(tree, point, k):
    indices, distances = tree.query(point, k)
    labels = tree.labels[indices]
    values, counts = np.unique(labels, return_counts=True)
    tied = values[counts == counts.max()]
    if len(tied) == 1:
        return int(tied[0]), indices, distances
    distance_sums = {int(label): float(distances[labels == label].sum()) for label in tied}
    best_distance = min(distance_sums.values())
    tied = [label for label in tied if np.isclose(distance_sums[label], best_distance)]
    prediction = int(labels[0]) if labels[0] in tied else min(tied)
    return prediction, indices, distances


k = 5
tree = KDTree(X_train, y_train, leaf_size=16)
predictions = np.array([classify(tree, row, k)[0] for row in X_test])
accuracy = float(np.mean(predictions == y_test))
tp = int(np.sum((y_test == 1) & (predictions == 1)))
tn = int(np.sum((y_test == 0) & (predictions == 0)))
fp = int(np.sum((y_test == 0) & (predictions == 1)))
fn = int(np.sum((y_test == 1) & (predictions == 0)))
print(f"Unique records: {len(data)}; k={k}; accuracy={accuracy:.3f}; TP={tp}; TN={tn}; FP={fp}; FN={fn}")
_, first_indices, first_distances = classify(tree, X_test[0], k)
print("First query neighbors (index, distance, class):")
print([(int(i), round(float(d), 3), int(y_train[i])) for i, d in zip(first_indices, first_distances)])
print("KD-tree nodes visited for one query:", tree.nodes_visited)

Unique records: 302; k=5; accuracy=0.803; TP=27; TN=22; FP=6; FN=6
First query neighbors (index, distance, class):
[(66, 2.235, 0), (225, 2.473, 0), (101, 2.641, 0), (159, 2.83, 0), (173, 2.869, 1)]
KD-tree nodes visited for one query: 31
